# TensorFly — real MaleCNS × Qwen3.5
Run this notebook top-to-bottom in a CUDA Colab runtime. It downloads the official MaleCNS v1.0 tables (about 1.1 GB) and never substitutes synthetic data.


In [ ]:
# 1. Clean-Colab setup and visible hardware/model decision
from pathlib import Path
import os, subprocess, sys
repo = Path('/content/fly-inference-optimizer')
if (repo / '.git').is_dir():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', 'main'], check=True)
elif repo.exists():
    raise RuntimeError(f'{repo} exists but is not a TensorFly clone; use a clean runtime.')
else:
    subprocess.run(['git', 'clone', '--branch', 'main', 'https://github.com/MrFaruk0/fly-inference-optimizer.git', str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'], check=True)
# Qwen3.5 requires current Transformers. Keep Colab's Torch/CUDA stack intact.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'git+https://github.com/huggingface/transformers.git', 'accelerate', 'safetensors', 'pyarrow', 'pillow', 'torchvision'], check=True)
source_root = str(repo / 'src')
if source_root not in sys.path: sys.path.insert(0, source_root)
import torch, tensorfly
if not torch.cuda.is_available(): raise RuntimeError('Select a CUDA runtime in Runtime → Change runtime type, then Run all again.')
gpu = torch.cuda.get_device_name(0); vram_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
try:
    import psutil; ram_gb = psutil.virtual_memory().total / 2**30
except ImportError:
    ram_gb = None
actual_model, fallback_notice = tensorfly.select_qwen_model(vram_gb=vram_gb)
print({'gpu': gpu, 'vram_gb': round(vram_gb, 1), 'ram_gb': None if ram_gb is None else round(ram_gb, 1), 'torch': torch.__version__, 'cuda': torch.version.cuda, 'requested_model': 'Qwen/Qwen3.5-9B', 'actual_model': actual_model})
if fallback_notice: print('MODEL FALLBACK:', fallback_notice)
# Optional acceleration only; failures remain visible and use correct reference kernels.
kernel = subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', 'causal-conv1d', 'flash-linear-attention'])
print('Optional Qwen kernel install exit code:', kernel.returncode)
print('TensorFly version:', tensorfly.__version__)


In [ ]:
# 2. Official MaleCNS v1.0: download/cache → SHA-256 validate → preprocess → real SWC viewer asset
prepared = tensorfly.prepare()
report = prepared.dataset.report
print({key: report.get(key) for key in ('release', 'retained_neurons', 'retained_edge_rows', 'synaptic_contact_count', 'retention_policy', 'preparation_timestamp')})
print({name: len(ids) for name, ids in prepared.populations.body_ids.items()})
print('Viewer morphology:', prepared.viewer_morphology)


In [ ]:
# 3. Measured equal-work Qwen experiment and baselines
from tensorfly import DEFAULT_PROMPTS, InferenceConfig, TensorFlyExperiment
batch_size = 2 if vram_gb >= 22 else 1
config = InferenceConfig(model_id=actual_model, batch_size=batch_size, max_new_tokens=32, dtype='bfloat16' if vram_gb >= 22 else 'float16')
experiment = TensorFlyExperiment(model=actual_model)
tensorfly_records = experiment.run(prompt_corpus=DEFAULT_PROMPTS, trials=20, config=config, warmup=1)
baselines = experiment.compare_baselines(prompt_corpus=DEFAULT_PROMPTS, trials=20, config=config, warmup=1)
replay_path = experiment.export_video()  # writes measured replay; browser performs recording after timings finish
summary = experiment.baseline_summary()
print('Actual model/runtime:', experiment.records[0].raw_inference_metrics.get('runtime', {}))
print('Equal-budget baseline summary:', summary)
print('Replay:', replay_path)


In [ ]:
# 4. Browser-native Three.js viewer in Colab. Click Record Demo → Stop Recording → Download Video.
import subprocess
server = subprocess.Popen([sys.executable, '-m', 'http.server', '8000', '--directory', 'viewer'])
from google.colab import output
output.serve_kernel_port_as_window(8000)
